In [6]:
import os, glob, cv2, random

# === CẤU HÌNH ===
TEST_DIR = r"E:\data_train\data_train\test"   # <-- sửa đường dẫn test/
OUTPUT   = r".\steel_demo_slideshow.mp4"
FPS      = 1                 # 1 frame/giây
SECONDS_PER_IMAGE = 5        # số giây hiển thị mỗi ảnh
REPEAT = SECONDS_PER_IMAGE   # lặp mỗi ảnh REPEAT lần

# Lấy danh sách ảnh
exts = ("*.jpg","*.jpeg","*.png","*.bmp")
img_paths = []
for e in exts:
    img_paths.extend(glob.glob(os.path.join(TEST_DIR, "*", e)))
img_paths = sorted(set(img_paths))
print(f"Found {len(img_paths)} images.")

# Kích thước
sample = cv2.imread(img_paths[0])
H,W = sample.shape[:2]

# Writer
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
vw = cv2.VideoWriter(OUTPUT, fourcc, FPS, (W,H))

# Ghép thành slideshow
for p in img_paths:
    img = cv2.imread(p)
    if img is None: continue
    if (img.shape[1], img.shape[0]) != (W,H):
        img = cv2.resize(img,(W,H))
    for _ in range(REPEAT):
        vw.write(img)

vw.release()
print(f"✅ Saved {OUTPUT} | {SECONDS_PER_IMAGE}s per image at {FPS} FPS")

Found 381 images.
✅ Saved .\steel_demo_slideshow.mp4 | 5s per image at 1 FPS


In [1]:
# !pip install opencv-python torch torchvision pillow numpy
import os, json, time, cv2, numpy as np
import torch, torch.nn as nn
from torchvision import models, transforms
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if DEVICE=="cuda" else "CPU")

# Đường dẫn
CKPT_PATH = "./outputs_cls_cam/resnet50_cls.pth"
MAP_PATH  = "./outputs_cls_cam/idx_to_class.json"
VIDEO_PATH = "./steel_demo_slideshow.mp4"   # <<— video bạn đã ghép 5s/ảnh
OUT_DIR = "./realtime_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# Load mapping lớp
with open(MAP_PATH, "r", encoding="utf-8") as f:
    idx_to_class = json.load(f)
NUM_CLASSES = len(idx_to_class)

# Model y hệt lúc train
def build_model(num_classes):
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    in_feats = m.fc.in_features
    m.fc = nn.Linear(in_feats, num_classes)
    return m

model = build_model(NUM_CLASSES).to(DEVICE).eval()

# Load checkpoint (hỗ trợ cả {"model": state_dict} hoặc state_dict thuần)
state = torch.load(CKPT_PATH, map_location=DEVICE)
state = state["model"] if isinstance(state, dict) and "model" in state else state
model.load_state_dict(state)
print("Loaded:", CKPT_PATH)

# Transform (để khớp với lúc train; nếu bạn train 224 thì đổi IMG_SIZE=224)
IMG_SIZE = 384
MEAN = [0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])


Device: NVIDIA GeForce RTX 3050 Laptop GPU
Loaded: ./outputs_cls_cam/resnet50_cls.pth


C:\Users\MINH\AppData\Local\Temp\ipykernel_23136\162024960.py:32: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(CKPT_PATH, map_location=DEVICE)


In [2]:
import torch.nn.functional as F

class CamPlusPlus:
    def __init__(self, model, target_layer_name="layer4"):
        self.model = model.eval()
        self.fmap = None; self.grad = None
        layer = dict([*model.named_modules()])[target_layer_name]
        self.h1 = layer.register_forward_hook(self._save_fmap)
        self.h2 = layer.register_full_backward_hook(self._save_grad)
    def _save_fmap(self, m, i, o): self.fmap = o.detach()
    def _save_grad(self, m, gi, go): self.grad = go[0].detach()
    def remove(self): self.h1.remove(); self.h2.remove()

    def __call__(self, x, class_idx=None):
        logits = self.model(x)
        if class_idx is None: class_idx = logits.argmax(1).item()
        score = logits[0, class_idx]
        self.model.zero_grad(set_to_none=True)
        score.backward(retain_graph=True)

        fmap = self.fmap[0]; grad = self.grad[0]
        grad2, grad3 = grad**2, grad**3
        eps = 1e-8
        sum_grad = grad.sum(dim=(1,2), keepdim=True)
        alpha = grad2 / (2*grad2 + (fmap*grad3).sum(dim=(1,2), keepdim=True) + eps)
        weights = (alpha * torch.relu(sum_grad)).sum(dim=(1,2))
        cam = torch.relu((weights.view(-1,1,1)*fmap).sum(0))
        cam = (cam - cam.min()) / (cam.max() - cam.min() + eps)
        return cam.cpu().numpy(), int(class_idx)

campp = CamPlusPlus(model, "layer4")

def heatmap_to_bboxes(heatmap, pct=87, min_area=120, min_wh=8):
    """Từ heatmap → mask nhị phân → contour → bbox (x,y,w,h)"""
    thr = np.percentile(heatmap, pct)
    m = (heatmap >= thr).astype(np.uint8)*255
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5))
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, k, 1)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k, 2)

    cnts,_ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    bboxes = []
    for c in cnts:
        if cv2.contourArea(c) < min_area: 
            continue
        x,y,w,h = cv2.boundingRect(c)
        if w < min_wh or h < min_wh: 
            continue
        bboxes.append((x,y,w,h))
    return bboxes  # chỉ trả bbox, không trả ảnh nhiệt


In [ ]:
import time
import numpy as np
import cv2
from PIL import Image
import torch

# --- tham số bạn có thể chỉnh nhanh ---
CONF_OK_THR      = 0.65   # < ngưỡng coi là "OK"
SCENE_DELTA_THR  = 6.0    # ngưỡng thay đổi khung để nhận bề mặt mới (MAD 0..255)
HOLD_SECONDS     = 5.0    # HIỂN THỊ 5 GIÂY mỗi lần
COOLDOWN_SECONDS = 4.5    # chống double-trigger khi ảnh giữ 5 giây
PCT              = 87     # percentile trích CAM để vẽ bbox

cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), f"Không mở được video: {VIDEO_PATH}"

last_key_small  = None
last_trigger_ts = 0.0

print("Press 'q' to quit.")
while True:
    ok, frame = cap.read()
    if not ok:
        break

    # === phát hiện bề mặt mới bằng so sánh khung ===
    small = cv2.resize(frame, (240, 135))
    gray  = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    gray  = cv2.GaussianBlur(gray, (5,5), 0)

    is_new_surface = False
    if last_key_small is None:
        is_new_surface = True
    else:
        diff = cv2.absdiff(gray, last_key_small)
        mad  = float(np.mean(diff))
        is_new_surface = mad > SCENE_DELTA_THR

    display = frame.copy()

    # chỉ trigger nếu vượt cooldown (tránh chụp nhiều lần cùng 1 bề mặt)
    if is_new_surface and (time.time() - last_trigger_ts >= COOLDOWN_SECONDS):
        last_key_small  = gray
        last_trigger_ts = time.time()
        stamp = time.strftime("%Y%m%d-%H%M%S")

        # === inference + quyết định OK / lỗi ===
        x = tf(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            logits = model(x)
            probs  = torch.softmax(logits, dim=1)[0].cpu().numpy()
            cls_idx = int(np.argmax(probs))
            cls_name = idx_to_class[str(cls_idx)]
            conf = float(probs[cls_idx])

        if conf < CONF_OK_THR:
            # KHÔNG lỗi
            cv2.putText(display, f"OK (no defect)  conf={conf:.2f}", (10,30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,200,255), 2)
            cv2.imwrite(os.path.join(OUT_DIR, f"{stamp}_OK.jpg"), display)
            print(f"[OK] conf={conf:.2f}")
        else:
            # Lỗi: dùng CAM++ để LẤY BBOX (không lưu ảnh nhiệt)
            hm, _ = campp(x, class_idx=cls_idx)
            hm = cv2.resize(hm, (frame.shape[1], frame.shape[0]), interpolation=cv2.INTER_CUBIC)

            bboxes = heatmap_to_bboxes(hm, pct=PCT, min_area=120, min_wh=8)
            vis = frame.copy()
            for (x1,y1,w,h) in bboxes:
                cv2.rectangle(vis, (x1,y1), (x1+w,y1+h), (0,255,0), 2)
            cv2.putText(vis, f"{cls_name}  conf={conf:.2f}", (10,30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)

            cv2.imwrite(os.path.join(OUT_DIR, f"{stamp}_{cls_name}_bbox.jpg"), vis)
            print(f"[DEFECT] {cls_name}  conf={conf:.2f}  bboxes={len(bboxes)}")
            display = vis

        # === GIỮ NGUYÊN KẾT QUẢ 5 GIÂY (demo realtime) ===
        end_t = time.time() + HOLD_SECONDS
        while time.time() < end_t:
            cv2.imshow("Steel QC demo", display)
            if cv2.waitKey(50) & 0xFF == ord('q'):
                end_t = 0; break
        # Sau khi giữ 5s, vòng lặp tiếp tục đọc video

    else:
        # Khi chưa có bề mặt mới: vẫn hiển thị khung hiện tại
        cv2.putText(display, f"waiting...(delta trigger={is_new_surface})", (10,30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)
        cv2.imshow("Steel QC demo", display)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release(); cv2.destroyAllWindows()
print("Done. Saved to:", OUT_DIR)


Press 'q' to quit.
[DEFECT] crease  conf=1.00  bboxes=2
[DEFECT] crease  conf=1.00  bboxes=4
[DEFECT] crease  conf=1.00  bboxes=2
[DEFECT] crease  conf=1.00  bboxes=1
[DEFECT] crease  conf=1.00  bboxes=4
[DEFECT] crease  conf=0.83  bboxes=2
[DEFECT] crease  conf=1.00  bboxes=2
[DEFECT] crease  conf=0.99  bboxes=3
[DEFECT] crease  conf=1.00  bboxes=3
[DEFECT] crease  conf=1.00  bboxes=2
[DEFECT] crease  conf=1.00  bboxes=3
[DEFECT] crease  conf=0.97  bboxes=4
[DEFECT] crease  conf=0.99  bboxes=4
[DEFECT] crease  conf=1.00  bboxes=2
[DEFECT] crease  conf=1.00  bboxes=1
[DEFECT] crease  conf=1.00  bboxes=3
[DEFECT] crease  conf=0.99  bboxes=1
[DEFECT] crease  conf=1.00  bboxes=3
[DEFECT] crease  conf=1.00  bboxes=2
[DEFECT] crease  conf=1.00  bboxes=2
[DEFECT] crease  conf=0.81  bboxes=2
[DEFECT] crease  conf=0.99  bboxes=3
[DEFECT] crescent_gap  conf=1.00  bboxes=3
[DEFECT] crescent_gap  conf=1.00  bboxes=4
[DEFECT] crescent_gap  conf=0.97  bboxes=2
[DEFECT] crescent_gap  conf=1.00  bbox